# 09-04 微调数据构造与清洗

**数据质量 > 数据数量**：1000条高质量数据 > 10000条低质量数据。

**本节目标**：构造 B站广告文案微调数据、数据清洗 pipeline、数据飞轮

---

In [ ]:
import json, random, hashlib
random.seed(42)

# 构造 B站广告文案 SFT 数据
PRODUCTS = [
    ("游戏皮肤", "限定游戏皮肤，多款角色可选", "游戏玩家"),
    ("大会员年卡", "1080P无广告、大会员番剧、每月B币", "动漫爱好者"),
    ("在线编程课", "Python/Java实战课程，名师授课", "大学生"),
    ("美妆新品", "夏日防晒新品，温和不刺激", "女性用户"),
    ("电商大促", "618全场优惠，限时折扣", "网购达人"),
]

TEMPLATES = [
    "为{product}写一个B站广告标题，15字以内，目标用户是{audience}",
    "生成{product}的广告文案，要求标题不超过15字，正文不超过50字",
    "针对{audience}群体，为{product}创作一个吸引眼球的广告标题",
]

GOOD_TITLES = {
    "游戏皮肤": ["沉浸游戏体验 限时畅玩", "游戏皮肤焕新 限时折扣", "你的角色值得更好的皮肤"],
    "大会员年卡": ["追番无广告 大会员畅享", "大会员年卡 尊享特权", "高清无广告 会员专属"],
    "在线编程课": ["零基础学编程 名师带飞", "Python实战课 轻松入门", "编程改变未来 现在开始"],
    "美妆新品": ["夏日防晒 温和守护", "美妆好物 闺蜜都推荐", "新品首发 限时体验价"],
    "电商大促": ["好物集结 优惠不等人", "品质好物 超值精选", "限时特惠 精选好物"],
}

# 生成数据集
dataset = []
for product, desc, audience in PRODUCTS:
    for template in TEMPLATES:
        for title in GOOD_TITLES[product]:
            instruction = template.format(product=product, audience=audience)
            dataset.append({
                "instruction": instruction,
                "input": f"产品描述: {desc}",
                "output": f"标题：{title}\n正文：{desc}，快来体验吧！",
                "product": product,
            })

random.shuffle(dataset)
print(f"生成数据集: {len(dataset)} 条")
print(f"\n示例:")
print(json.dumps(dataset[0], ensure_ascii=False, indent=2))

In [ ]:
# 数据清洗 pipeline
def clean_dataset(data: list[dict]) -> list[dict]:
    """数据清洗: 去重 → 长度过滤 → 质量检查 → 格式验证"""
    # 1. 去重（基于 instruction+output 哈希）
    seen = set()
    deduped = []
    for item in data:
        key = hashlib.md5((item['instruction'] + item['output']).encode()).hexdigest()
        if key not in seen:
            seen.add(key)
            deduped.append(item)
    print(f"  去重: {len(data)} → {len(deduped)}")
    
    # 2. 长度过滤
    length_ok = [d for d in deduped if 10 < len(d['instruction']) < 500 and 5 < len(d['output']) < 1000]
    print(f"  长度过滤: {len(deduped)} → {len(length_ok)}")
    
    # 3. 质量检查（极限词）
    forbidden = ['最', '第一', '绝对', '100%']
    quality_ok = [d for d in length_ok if not any(w in d['output'] for w in forbidden)]
    print(f"  质量检查: {len(length_ok)} → {len(quality_ok)}")
    
    # 4. 格式验证
    valid = [d for d in quality_ok if all(k in d for k in ['instruction', 'output'])]
    print(f"  格式验证: {len(quality_ok)} → {len(valid)}")
    
    return valid

print("=== 数据清洗 ===")
cleaned = clean_dataset(dataset)

# Train/Val 分割
split_idx = int(len(cleaned) * 0.9)
train_data = cleaned[:split_idx]
val_data = cleaned[split_idx:]
print(f"\n训练集: {len(train_data)}, 验证集: {len(val_data)}")

In [ ]:
# 数据飞轮概念
print("""
数据飞轮 (Data Flywheel):

  用户使用 Agent → 收集反馈 → 筛选高质量数据 → 微调模型 → 更好的 Agent → 更多用户
       ↑                                                                    ↓
       └────────────────────────────────────────────────────────────────────┘

B站广告场景:
  1. Agent 生成广告文案 → 广告主反馈(采纳/修改/拒绝)
  2. 采纳的文案 → 正样本，拒绝的 → 负样本
  3. 定期用新数据微调 → 模型越来越懂B站风格
  4. A/B测试: 微调模型 vs 基础模型，比较 CTR

关键指标:
  - 文案采纳率: 目标 > 70%
  - A/B测试 CTR 提升: 目标 > 10%
  - 每月新增高质量数据: 目标 > 500 条
""")

## 面试速记

| 问题 | 要点 |
|------|------|
| 微调数据最重要的是什么 | 质量 > 数量；多样性；与目标任务一致 |
| 需要多少数据 | SFT: 1K-10K 条即可；关键是覆盖所有场景 |
| 数据飞轮 | 用户反馈 → 数据 → 微调 → 更好服务 → 更多反馈，正循环 |
| 如何避免灾难性遗忘 | 混入通用数据、LoRA(不改原始权重)、小学习率 |